# 04 - Hypotheses, features and leakage audit

The hypotheses (H1 momentum/reversal, H2 volume/order flow, H3 regime dependence, H4 combination, H5 economic value) were **pre-registered before the data was seen**; see `reports/research_design_preregistration.md`.

**Features:** 16 scale-free features in five groups (returns, momentum, volatility, volume/order flow, candle). Each is computed only from candles that closed by the decision time *t*.

**Leakage audit:**
1. A feature-timing table: feature → latest information used → prediction time → execution time.
2. A perturbation test on the real data: replace everything at or after *t* with random values and recompute. A feature that changes would have been looking into the future.

**H1/H2 test:** Spearman rank correlation between each feature and the next-hour return on development data, a block-bootstrap p-value, and Holm correction across all 16 features.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
import run_pipeline as rp
from src import config
T, F = config.TABLES_DIR, config.FIGURES_DIR
SYMBOL = config.SYMBOL
# False (default): display the outputs saved by scripts/run_pipeline.py.
# True re-computes the stage. Do NOT re-run 'validation' or 'test' casually:
# validation resets the experiment log and every test run adds a TEST row.
RUN_STAGE = False
def show(name, **kw):
    display(Markdown(f"**{name}**")); display(pd.read_csv(T / name, **kw))
def fig(name):
    display(Image(filename=str(F / name)))
def results(section):
    return json.loads((config.REPORTS_DIR / "results.json").read_text()).get(section, {})

In [2]:
if RUN_STAGE:
    rp.stage_features(SYMBOL)

In [3]:
show('feature_timing_table.csv')

**feature_timing_table.csv**

,feature,meaning,lookback_minutes,information_window,latest_information,prediction_time,execution_time,target_window
0,ret_1m,log return over the last 1 minute,1,"candles opened in [t-1m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
1,ret_5m,log return over the last 5 minutes,5,"candles opened in [t-5m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
2,ret_15m,log return over the last 15 minutes,15,"candles opened in [t-15m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
3,ret_30m,log return over the last 30 minutes,30,"candles opened in [t-30m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
4,ret_60m,log return over the last 60 minutes,60,"candles opened in [t-60m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
5,ret_4h,log return over the last 4 hours,240,"candles opened in [t-240m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
6,ret_24h,log return over the last 24 hours,1440,"candles opened in [t-1440m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
7,ma_dist_24h,log distance of price from its 24h moving average,1440,"candles opened in [t-1440m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
8,log_rv_60m,log realised volatility of the last 60 minutes,60,"candles opened in [t-60m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"
9,log_rv_24h,log realised volatility of the last 24 hours (...,1440,"candles opened in [t-1440m, t-1m]",close of candle t-1m (closes exactly at t),t,open of candle t+1m,"[t+1m, t+61m]"


In [4]:
show('leakage_perturbation_real_data.csv')

**leakage_perturbation_real_data.csv**

,decision_time,features_changed,changed_list,target_changed
0,2018-07-03 02:00:00+00:00,0,NaN,True
1,2018-07-15 05:00:00+00:00,0,NaN,True
2,2018-07-31 09:00:00+00:00,0,NaN,True
3,2018-11-21 22:00:00+00:00,0,NaN,True
4,2019-05-21 20:00:00+00:00,0,NaN,True
5,2019-07-22 21:00:00+00:00,0,NaN,True
6,2021-02-04 12:00:00+00:00,0,NaN,True
7,2021-08-30 19:00:00+00:00,0,NaN,True
8,2021-09-18 23:00:00+00:00,0,NaN,True
9,2021-10-27 05:00:00+00:00,0,NaN,True


In [5]:
show('h1_h2_univariate_dev.csv')

**h1_h2_univariate_dev.csv**

,feature,spearman,p_block,naive_p,n,p_value,p_holm,reject_at_5pct
0,ret_1m,-0.007614,0.084915,7.332368e-02,55313.0,0.084915,0.169830,False
1,ret_5m,-0.046269,0.000999,1.321192e-27,55313.0,0.000999,0.015984,True
2,ret_15m,-0.066485,0.000999,3.164553e-55,55311.0,0.000999,0.015984,True
3,ret_30m,-0.074143,0.000999,2.934197e-68,55301.0,0.000999,0.015984,True
4,ret_60m,-0.072752,0.000999,9.241010e-66,55281.0,0.000999,0.015984,True
5,ret_4h,-0.067434,0.000999,1.105708e-56,55228.0,0.000999,0.015984,True
6,ret_24h,-0.022146,0.000999,1.984494e-07,55141.0,0.000999,0.015984,True
7,ma_dist_24h,-0.037529,0.000999,1.385264e-18,54930.0,0.000999,0.015984,True
8,log_rv_60m,0.018046,0.000999,2.198062e-05,55296.0,0.000999,0.015984,True
9,log_rv_24h,0.010413,0.011988,1.467511e-02,54916.0,0.011988,0.035964,True


In [6]:
show('feature_summary.csv')

**feature_summary.csv**

,Unnamed: 0,count,mean,std,min,25%,50%,75%,max
0,ret_1m,79464.0,0.000003,0.001012,-0.028460,-0.000274,0.000000,0.000281,0.026540
1,ret_5m,79464.0,0.000007,0.002129,-0.107868,-0.000652,0.000000,0.000670,0.068415
2,ret_15m,79462.0,0.000018,0.003690,-0.100827,-0.001196,0.000013,0.001249,0.083732
3,ret_30m,79452.0,0.000021,0.005354,-0.181873,-0.001729,0.000039,0.001804,0.159571
4,ret_60m,79432.0,0.000029,0.007571,-0.201033,-0.002420,0.000065,0.002579,0.160280
5,ret_4h,79378.0,0.000120,0.014403,-0.342863,-0.004659,0.000186,0.005155,0.271624
6,ret_24h,79288.0,0.000732,0.035016,-0.624510,-0.013596,0.000847,0.015696,0.298704
7,ma_dist_24h,79079.0,0.000279,0.020015,-0.438335,-0.007126,0.000492,0.008300,0.223259
8,log_rv_60m,79447.0,-5.349739,0.707205,-8.692997,-5.813541,-5.383597,-4.916109,-1.627836
9,log_rv_24h,79113.0,-5.181817,0.587890,-7.648676,-5.547791,-5.216206,-4.856977,-2.600202


**Interpretation:** see the matching section of `reports/final_report.md`, written from these outputs.